In [650]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

In [651]:
seed = 1

### Neural data generator

In [652]:
class GroundTruthNN(nn.Module):
    def __init__(self, x_dim, h_dim, y_dim, hidden_layers=[64, 64]):
        super().__init__()
        self.x_dim = x_dim
        self.h_dim = h_dim
        self.input_dim = x_dim + h_dim

        layers = []
        in_dim = self.input_dim

        # Hidden layers (keep tanh)
        for h in hidden_layers:
            layers.append(nn.Linear(in_dim, h))
            #layers.append(nn.Tanh())
            in_dim = h

        # Output layer + sigmoid
        layers.append(nn.Linear(in_dim, y_dim))
        layers.append(nn.Sigmoid())

        self.net = nn.Sequential(*layers)

    def forward(self, x, h):
        z = torch.cat([x, h], dim=1)
        return self.net(z)


In [653]:
"""
class GroundTruthNN(nn.Module):
    def __init__(self, x_dim, h_dim, y_dim, hidden_layers=[64, 64]):
        super().__init__()
        self.x_dim = x_dim
        self.h_dim = h_dim
        self.input_dim = x_dim + h_dim
        layers = []
        in_dim = self.input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.Tanh())  # smooth nonlinearity (stable)
            in_dim = h
        layers.append(nn.Linear(in_dim, y_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x, h):
        z = torch.cat([x, h], dim=1)
        return self.net(z)
"""
   
   
def create_generator(x_dim, h_dim, y_dim):
    model = GroundTruthNN(x_dim, h_dim, y_dim)
    # Freeze parameters (important!)
    for param in model.parameters():
        param.requires_grad = False
    return model

In [654]:
def generate_dataset_with_domains(generator, x, h, domain_ids):
    """
    generator : GroundTruthNN (frozen)
    x : (n_samples, x_dim)
    h : (n_samples, h_dim)
    domain_ids : (n_samples,) array-like
    Returns:
        pandas.DataFrame with columns:
        x_i, h_j, y_k, domain
    """
    if not isinstance(x, torch.Tensor):
        x = torch.tensor(x, dtype=torch.float32)
    if not isinstance(h, torch.Tensor):
        h = torch.tensor(h, dtype=torch.float32)
    domain_ids = np.asarray(domain_ids)
    if len(domain_ids) != x.shape[0]:
        raise ValueError("domain_ids must have same length as number of samples")
    with torch.no_grad():
        y = generator(x, h)
    x_np = x.numpy()
    h_np = h.numpy()
    y_np = y.numpy()
    data_dict = {}
    # x columns
    for i in range(x_np.shape[1]):
        data_dict[f"x_{i}"] = x_np[:, i]
    # h columns
    for j in range(h_np.shape[1]):
        data_dict[f"h_{j}"] = h_np[:, j]
    # y columns
    for k in range(y_np.shape[1]):
        data_dict[f"y_{k}"] = y_np[:, k]
    # domain column
    data_dict["domain"] = domain_ids
    df = pd.DataFrame(data_dict)
    return df

In [655]:
def generate_domain_samples(
    n_samples,
    x_dim,
    h_dim,
    x_mean,
    x_std,
    h_mean,
    h_std,
    domain_id
):
    """
    x_mean, x_std : array-like of shape (x_dim,)
    h_mean, h_std : array-like of shape (h_dim,)
    """
 
    # Convert to tensors
    x_mean = torch.tensor(x_mean, dtype=torch.float32)
    x_std = torch.tensor(x_std, dtype=torch.float32)
    h_mean = torch.tensor(h_mean, dtype=torch.float32)
    h_std = torch.tensor(h_std, dtype=torch.float32)
 
    # Safety checks
    if len(x_mean) != x_dim or len(x_std) != x_dim:
        raise ValueError("x_mean and x_std must have length x_dim")
 
    if len(h_mean) != h_dim or len(h_std) != h_dim:
        raise ValueError("h_mean and h_std must have length h_dim")
 
    # Gaussian sampling per feature
    x = torch.randn(n_samples, x_dim) * x_std + x_mean
    h = torch.randn(n_samples, h_dim) * h_std + h_mean
 
    domain_ids = np.full(n_samples, domain_id)
 
    return x, h, domain_ids

In [656]:
def simulate_multiple_domains(
    total_samples,
    x_dim,
    h_dim,
    domain_params_list
):
    """
    Simulates multiple balanced domains.
    Parameters
    ----------
    total_samples : int
    x_dim : int
    h_dim : int
    domain_params_list : list of dict
        Each dict must contain:
        {
            "x_mean": ...,
            "x_std": ...,
            "h_mean": ...,
            "h_std": ...
        }
    Returns
    -------
    x_all : np.ndarray
    h_all : np.ndarray
    domain_ids_all : np.ndarray
    """
    n_domains = len(domain_params_list)
    if total_samples % n_domains != 0:
        raise ValueError("total_samples must be divisible by number of domains for balance.")
    samples_per_domain = total_samples // n_domains
    x_list = []
    h_list = []
    domain_list = []
    for domain_id, params in enumerate(domain_params_list):
        x_d, h_d, d_d = generate_domain_samples(
            n_samples=samples_per_domain,
            x_dim=x_dim,
            h_dim=h_dim,
            x_mean=params["x_mean"],
            x_std=params["x_std"],
            h_mean=params["h_mean"],
            h_std=params["h_std"],
            domain_id=domain_id
        )
        x_list.append(x_d.numpy())
        h_list.append(h_d.numpy())
        domain_list.append(d_d)
    # Concatenate all domains
    x_all = np.vstack(x_list)
    h_all = np.vstack(h_list)
    domain_ids_all = np.concatenate(domain_list)
    return x_all, h_all, domain_ids_all

In [657]:
def build_synthetic_dataset(
    total_samples,
    x_dim,
    h_dim,
    y_dim,
    domain_params_list,
    hidden_layers=[64, 64],
    random_seed=None
):
    """
    Full synthetic data generation pipeline.
    Returns:
    - df : pandas DataFrame containing x, h, y, domain
    - generator : the frozen GroundTruthNN used
    """
    if random_seed is not None:
        import torch
        import numpy as np
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
    # 1️ Create fixed ground-truth generator
    generator = create_generator(x_dim, h_dim, y_dim)
    # 2️ Simulate domains (x, h, domain ids)
    x_all, h_all, domain_ids = simulate_multiple_domains(
        total_samples=total_samples,
        x_dim=x_dim,
        h_dim=h_dim,
        domain_params_list=domain_params_list
    )
    # 3️ Generate outputs and assemble dataset
    df = generate_dataset_with_domains(
        generator=generator,
        x=x_all,
        h=h_all,
        domain_ids=domain_ids
    )
    return df, generator

In [658]:
def create_domain_params(
    n_domains,
    x_dim,
    h_dim,
    x_mean_range=(0.0, 0.0),
    h_mean_range=(0.0, 0.0),
    x_mean_dist_range=(0.0, 0.0),
    h_mean_dist_range=(0.0, 0.0),
    x_std_range=(1.0, 1.0),
    h_std_range=(0.0, 0.0),
    random_seed=None,
    max_attempts=50000
):
    """
    Generates domain parameter list with:
    - Component-wise mean bounds
    - Pairwise mean distance bounds
    - Std sampled from value ranges
    All ranges are (min, max).
    """
 
    if random_seed is not None:
        np.random.seed(random_seed)
 
    def generate_means(dim, value_range, dist_range):
        means = []
        min_val, max_val = value_range
        min_dist, max_dist = dist_range
 
        for _ in range(n_domains):
            for _ in range(max_attempts):
                candidate = np.random.uniform(min_val, max_val, size=dim)
 
                if len(means) == 0:
                    means.append(candidate)
                    break
 
                distances = [np.linalg.norm(candidate - m) for m in means]
 
                if all(min_dist <= d <= max_dist for d in distances):
                    means.append(candidate)
                    break
            else:
                raise RuntimeError("Could not generate means satisfying constraints.")
 
        return means
 
    # Generate means
    x_means = generate_means(x_dim, x_mean_range, x_mean_dist_range)
    h_means = generate_means(h_dim, h_mean_range, h_mean_dist_range)
 
    params = []
 
    for i in range(n_domains):
 
        x_std = np.random.uniform(x_std_range[0], x_std_range[1], size=x_dim)
        h_std = np.random.uniform(h_std_range[0], h_std_range[1], size=h_dim)
 
        params.append({
            "x_mean": x_means[i],
            "x_std": x_std,
            "h_mean": h_means[i],
            "h_std": h_std,
        })
 
    return params

### No Shift

In [659]:
x_dim = 50
h_dim = 10
y_dim = 1
n_domains = 10
total_samples = 5000
x_mean_range=(-5.0, 5.0)
h_mean_range=(-5.0, 5.0)
x_std_range=(10.0, 15.0)

In [660]:
no_params = create_domain_params(
    n_domains=1,
    x_dim=x_dim,
    h_dim=h_dim,
    x_mean_range=x_mean_range,
    h_mean_range=h_mean_range,
    x_std_range=x_std_range
)
 
df_no, generator = build_synthetic_dataset(
    total_samples=total_samples,
    x_dim=x_dim,
    h_dim=h_dim,
    y_dim=y_dim,
    domain_params_list=no_params,
    random_seed=seed
)
 
df_no.to_csv("data_generated/no_shift_df.csv", index=False)


### Sampling Shift

In [661]:
x_mean_range=(-10.0, 10.0)
x_std_range=(5.5, 5.5)
x_mean_dist_range=(5.0, 10000.0)

In [662]:
sampling_params = create_domain_params(
    n_domains=n_domains,
    x_dim=x_dim,
    h_dim=h_dim,
    x_mean_range=x_mean_range,
    x_mean_dist_range=x_mean_dist_range,
    x_std_range=x_std_range
)
 
df_sampling, generator = build_synthetic_dataset(
    total_samples=total_samples,
    x_dim=x_dim,
    h_dim=h_dim,
    y_dim=y_dim,
    domain_params_list=sampling_params,
    random_seed=seed
)
 
df_sampling.to_csv("data_generated/sampling_shift_df.csv", index=False)

### Hidden Shift

In [663]:
h_mean_range=(-10.0, 10.0)
h_mean_dist_range=(10.0, 100.0)
x_std_range=(2.0, 4.0)
h_std_range=(0.1, 0.5)

In [664]:
hidden_params = create_domain_params(
    n_domains=n_domains,
    x_dim=x_dim,
    h_dim=h_dim,
    h_mean_range=h_mean_range,
    x_std_range=x_std_range,
    h_mean_dist_range=h_mean_dist_range,
    h_std_range=h_std_range
)
 
df_hidden, generator = build_synthetic_dataset(
    total_samples=total_samples,
    x_dim=x_dim,
    h_dim=h_dim,
    y_dim=y_dim,
    domain_params_list=hidden_params,
    random_seed=seed
)
 
df_hidden.to_csv("data_generated/hidden_shift_df.csv", index=False)

### Observer Shift

In [665]:
def apply_observer_domains(
    df,
    n_domains,
    z_dim,
    x_prefix="x_",
    random_seed=None,
    noise_scale=0.5,
    perturb_ratio = 0.5
):
    if random_seed is not None:
        np.random.seed(random_seed)

    df = df.copy()

    # Identify x columns
    x_cols = sorted(
        [c for c in df.columns if c.startswith(x_prefix)],
        key=lambda x: int(x.split("_")[1])
    )

    x_dim = len(x_cols)
    total_samples = len(df)

    if total_samples % n_domains != 0:
        raise ValueError("Number of samples must be divisible by n_domains.")

    samples_per_domain = total_samples // n_domains

    #projections = {
    #    d: np.random.randn(z_dim, x_dim) / np.sqrt(x_dim)
    #    for d in range(n_domains)
    #}
    
    # Base projection (shared across domains)
    
    #W0 = np.random.randn(z_dim, x_dim) / np.sqrt(x_dim)

    #projections = {
    #    d: W0 + noise_scale * (
    #        np.random.randn(z_dim, x_dim) / np.sqrt(x_dim)
    #    )
    #    for d in range(n_domains)
    #}
    
    
    W0 = np.random.randn(z_dim, x_dim) / np.sqrt(x_dim)

    n_perturb = int(z_dim * perturb_ratio)

    # Choose ONCE → globally domain-variant rows
    perturb_rows = np.random.choice(z_dim, n_perturb, replace=False)

    projections = {}

    for d in range(n_domains):
        Wd = W0.copy()

        noise = np.random.randn(n_perturb, x_dim) / np.sqrt(x_dim)
        Wd[perturb_rows] += noise_scale * noise

        projections[d] = Wd

    
    


    df_list = []

    for d in range(n_domains):
        start = d * samples_per_domain
        end = (d + 1) * samples_per_domain

        df_chunk = df.iloc[start:end].copy()

        x_values = df_chunk[x_cols].values
        W = projections[d]
        z_values = x_values @ W.T

        # Remove old x columns
        df_chunk = df_chunk.drop(columns=x_cols)

        # Add transformed x columns
        for j in range(z_values.shape[1]):
            df_chunk[f"x_{j}"] = z_values[:, j]

        # Assign domain
        df_chunk["domain"] = d

        df_list.append(df_chunk)

    df_new = pd.concat(df_list, ignore_index=True)

    # Reorder columns: x first, then h, then y, then domain
    x_new_cols = sorted(
        [c for c in df_new.columns if c.startswith("x_")],
        key=lambda x: int(x.split("_")[1])
    )
    h_cols = sorted([c for c in df_new.columns if c.startswith("h_")])
    y_cols = sorted([c for c in df_new.columns if c.startswith("y_")])

    df_new = df_new[x_new_cols + h_cols + y_cols + ["domain"]]

    return df_new


In [666]:
df_observed = apply_observer_domains(
    df=df_no,
    n_domains=n_domains,
    z_dim=x_dim,      # choose latent dimension
    random_seed=seed
)

df_observed.to_csv("data_generated/observer_shift_df.csv", index=False)